In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import os
import copy
import random
import math
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from tqdm import trange

from data_utils import load_and_prepare_data, build_spatiotemporal_dataset
from config import PREDICTION_HORIZONS, LOOKBACK_DAYS, OUT_DIR

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## Configuration

In [ ]:
CONFIG = {
    'data_file': "../data/repeaters.csv",
    'epochs': 100,
    'batch_size': 16,
    'out_dir': OUT_DIR,
    'lookback_months': int(LOOKBACK_DAYS/30),
    'temporal_dim': 3,
    'hidden_dim': 16,
    'num_layers': 2,
    'num_heads': 4,
    'num_horizons': len(PREDICTION_HORIZONS),
    'dropout': 0.4,
    'lr': 3e-4,
    'weight_decay': 1e-2
}

file_name = os.path.splitext(os.path.basename(CONFIG['data_file']))[0]
print(f"Data file: {CONFIG['data_file']}")
print(f"Prediction horizons: {PREDICTION_HORIZONS} days")

## Data Loading and Preprocessing

In [ ]:
def collate_fn(batch):
    return {
        'x_spatial': torch.stack([b.x_spatial for b in batch]),
        'x_temporal': torch.stack([b.x_temporal for b in batch]),
        'y': torch.stack([b.y for b in batch]),
        'dist_matrix': torch.stack([b.dist_matrix for b in batch]),
    }


def prescale_data(data_list, scaler=None, fit=False):
    """
    Pre-scale temporal features and convert spatial to LongTensor.
    """
    if fit:
        print("Fitting scaler on training data")
        all_temporal = []
        for d in data_list:
            all_temporal.append(d.x_temporal.view(-1, 3).numpy())
        all_temporal = np.concatenate(all_temporal, axis=0)
        scaler = RobustScaler()
        scaler.fit(all_temporal)
    
    for d in data_list:
        # Scale temporal
        N, L, C = d.x_temporal.shape
        t_flat = d.x_temporal.view(-1, C).numpy()
        t_scaled = scaler.transform(t_flat)
        d.x_temporal = torch.tensor(t_scaled, dtype=torch.float32).view(N, L, C)
        # Convert spatial to Long
        d.x_spatial = d.x_spatial.view(-1).long()
    
    return scaler

In [ ]:
print(f"Loading data from {CONFIG['data_file']}")
df = load_and_prepare_data(CONFIG['data_file'])

print("\nBuilding spatiotemporal dataset...")
data_list, _ = build_spatiotemporal_dataset(df, lookback_months=CONFIG['lookback_months'])

# Split chronologically into train/val/test (70/15/15)
n_total = len(data_list)
n_train = int(0.7 * n_total)
n_val = int(0.85 * n_total)

train_raw = data_list[:n_train]
val_raw = data_list[n_train:n_val]
test_raw = data_list[n_val:]

print(f"\nSplit: Train={len(train_raw)} snapshots, Val={len(val_raw)} snapshots, Test={len(test_raw)} snapshots")

# Pre-scale data (fit on train, apply to all)
t_scaler = prescale_data(train_raw, fit=True)
prescale_data(val_raw, scaler=t_scaler, fit=False)
prescale_data(test_raw, scaler=t_scaler, fit=False)

train_loader = DataLoader(train_raw, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_raw, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_raw, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

num_nodes = train_raw[0].x_spatial.shape[0]
print(f"Number of nodes: {num_nodes}")

## Heterogeneous Transformer Model

In [ ]:
class DistanceBias(nn.Module):
    """Computes distance bias for spatial attention."""
    def __init__(self, tau_km: float = 25.0):
        super().__init__()
        self.tau = float(tau_km)

    def forward(self, dist_matrix: torch.Tensor):
        if dist_matrix.dim() == 2:
            dist_matrix = dist_matrix.unsqueeze(0)
        # bias = -torch.clamp(dist_matrix, min=0.0) / self.tau
        max_dist = torch.max(dist_matrix)
        bias = 1.0 - (dist_matrix / (max_dist + 1e-8))
        return bias.unsqueeze(1)  # (B,1,N,N)


class SpatialAttention(nn.Module):
    """Multi-head attention between spatial nodes with distance bias."""
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.qkv_proj = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, bias: torch.Tensor):
        B, N, C = x.shape
        h = self.norm(x)
        
        qkv = self.qkv_proj(h)
        q, k, v = qkv.chunk(3, dim=-1)
        
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        scores = scores +  bias
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        out = self.out_proj(out)
        
        return x + self.dropout(out)


class TemporalToSpatialAttention(nn.Module):
    """Multi-head message passing from temporal nodes to their corresponding spatial node."""
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.kv_proj = nn.Linear(hidden_dim, 2 * hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, spatial_nodes: torch.Tensor, temporal_nodes: torch.Tensor):
        B, N, C = spatial_nodes.shape
        L = temporal_nodes.shape[2]
        
        q = self.q_proj(spatial_nodes)
        kv = self.kv_proj(temporal_nodes)
        k, v = kv.chunk(2, dim=-1)
        
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2).unsqueeze(3)
        k = k.view(B, N, L, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)
        v = v.view(B, N, L, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)
        
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v).squeeze(3)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        out = self.out_proj(out)
        
        return out


class TransformerBlock(nn.Module):
    """One block of transformer message passing."""
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1, use_temporal: bool = False):
        super().__init__()
        self.use_temporal = use_temporal
        if use_temporal:
            self.temporal_to_spatial = TemporalToSpatialAttention(hidden_dim, num_heads, dropout)
            self.proj_ln = nn.LayerNorm(hidden_dim * 2)
            self.proj_lin = nn.Linear(hidden_dim * 2, hidden_dim)

        self.spatial_attn = SpatialAttention(hidden_dim, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, h_spatial: torch.Tensor, h_temporal: torch.Tensor, dist_bias: torch.Tensor):
        if self.use_temporal:
            t_msg = self.temporal_to_spatial(h_spatial, h_temporal)
            h = torch.cat([h_spatial, t_msg], dim=-1)
            h = self.proj_ln(h)
            h = self.proj_lin(h)
        else:
            h = h_spatial

        h = self.spatial_attn(h, dist_bias)
        h = h + self.ffn(h)
        return h


class HeterogeneousTransformer(nn.Module):
    """Heterogeneous graph transformer for earthquake prediction."""
    def __init__(self, num_nodes: int, temporal_feat_dim: int = 3, hidden_dim: int = 64,
                 num_heads: int = 4, num_layers: int = 2, num_horizons: int = 3, 
                 dropout: float = 0.1, tau_km: float = 25.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_nodes = num_nodes
        self.num_layers = num_layers

        self.spatial_embedding = nn.Embedding(num_nodes, hidden_dim)
        self.temporal_encoder = nn.Sequential(
            nn.Linear(temporal_feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.temporal_pos_encoding = nn.Embedding(100, hidden_dim)
        self.distance_bias = DistanceBias(tau_km=tau_km)

        layers = []
        for i in range(num_layers):
            use_temporal = (i == 0)
            layers.append(TransformerBlock(hidden_dim, num_heads, dropout, use_temporal=use_temporal))
        self.layers = nn.ModuleList(layers)

        self.final_norm = nn.LayerNorm(hidden_dim)
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            ) for _ in range(num_horizons)
        ])

    def forward(self, x_spatial: torch.Tensor, x_temporal: torch.Tensor, dist_matrix: torch.Tensor):
        if x_spatial.dim() == 3:
            x_spatial = x_spatial.squeeze(-1)
        
        B, N, L, C = x_temporal.shape
        
        h_spatial = self.spatial_embedding(x_spatial.long())
        h_temporal = self.temporal_encoder(x_temporal)
        
        positions = torch.arange(L, device=x_temporal.device)
        pos_enc = self.temporal_pos_encoding(positions)
        h_temporal = h_temporal + pos_enc.unsqueeze(0).unsqueeze(0)
        
        dist_bias = self.distance_bias(dist_matrix)

        h = h_spatial
        for layer in self.layers:
            h = layer(h, h_temporal, dist_bias)

        h = self.final_norm(h)
        out = torch.stack([head(h).squeeze(-1) for head in self.heads], dim=-1)
        return out

In [ ]:
class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance."""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_weight * focal_weight * bce_loss
        return loss.mean()

## Training and Evaluation Functions

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion):
    """Train for one epoch."""
    model.train()
    train_losses = []
    
    for batch in train_loader:
        x_s = batch['x_spatial'].to(DEVICE)
        x_t = batch['x_temporal'].to(DEVICE)
        y = batch['y'].to(DEVICE)
        dist = batch['dist_matrix'].to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(x_s, x_t, dist)
        loss = criterion(logits, y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_losses.append(loss.item())
    
    return np.mean(train_losses) if train_losses else 0.0


@torch.no_grad()
def evaluate(model, data_loader, criterion):
    """Evaluate model and return metrics."""
    model.eval()
    all_logits = []
    all_targets = []
    all_losses = []
    
    for batch in data_loader:
        x_s = batch['x_spatial'].to(DEVICE)
        x_t = batch['x_temporal'].to(DEVICE)
        y = batch['y'].to(DEVICE)
        dist = batch['dist_matrix'].to(DEVICE)
        
        logits = model(x_s, x_t, dist)
        loss = criterion(logits, y)
        
        all_logits.append(logits.cpu())
        all_targets.append(y.cpu())
        all_losses.append(loss.item())
    
    if len(all_logits) == 0:
        return {}, 0.0, 0.0
    
    all_logits = torch.cat(all_logits, dim=0).view(-1, len(PREDICTION_HORIZONS)).numpy()
    all_targets = torch.cat(all_targets, dim=0).view(-1, len(PREDICTION_HORIZONS)).numpy()
    all_probs = 1 / (1 + np.exp(-all_logits))
    
    # Calculate metrics per horizon
    metrics = {}
    aucs = []
    for i, horizon in enumerate(PREDICTION_HORIZONS):
        if len(np.unique(all_targets[:, i])) > 1:
            auc = roc_auc_score(all_targets[:, i], all_probs[:, i])
            aucs.append(auc)
            
            # Find best threshold
            best_f1 = 0
            best_thresh = 0.5
            for thresh in np.linspace(0.1, 0.9, 17):
                preds = (all_probs[:, i] > thresh).astype(int)
                _, _, f1, _ = precision_recall_fscore_support(
                    all_targets[:, i], preds, average="binary", zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_thresh = thresh
            
            preds = (all_probs[:, i] > best_thresh).astype(int)
            acc = accuracy_score(all_targets[:, i], preds)
            prec, rec, f1, _ = precision_recall_fscore_support(
                all_targets[:, i], preds, average="binary", zero_division=0)
            
            metrics[horizon] = {
                'auc': auc,
                'accuracy': acc,
                'precision': prec,
                'recall': rec,
                'f1': f1,
                'threshold': best_thresh
            }
        else:
            aucs.append(0)
            metrics[horizon] = {'auc': 0, 'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0, 'threshold': 0.5}
    
    avg_auc = np.mean(aucs) if aucs else 0
    avg_loss = np.mean(all_losses)
    
    return metrics, avg_auc, avg_loss

## Model Initialization

In [ ]:
model = HeterogeneousTransformer(
    num_nodes=num_nodes,
    temporal_feat_dim=CONFIG['temporal_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    num_horizons=len(PREDICTION_HORIZONS),
    dropout=CONFIG['dropout'],
    tau_km=25.0
).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
criterion = FocalLoss(alpha=0.70, gamma=2.0)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {model.__class__.__name__}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Training Loop

In [ ]:
best_model = None
best_val_auc = 0

# Storage for plotting
train_loss_history = []
val_loss_history = []
scores_train = []
scores_val = []
scores_test = []

print("Starting Training...")

for epoch in range(1, CONFIG['epochs'] + 1):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    train_loss_history.append(train_loss)
    
    # Evaluate on all sets
    train_metrics, train_auc, _ = evaluate(model, train_loader, criterion)
    val_metrics, val_auc, val_loss = evaluate(model, val_loader, criterion)
    test_metrics, test_auc, _ = evaluate(model, test_loader, criterion)
    
    val_loss_history.append(val_loss)
    
    # Compute F1 scores
    train_f1 = np.mean([train_metrics[h]['f1'] for h in PREDICTION_HORIZONS]) if train_metrics else 0
    val_f1 = np.mean([val_metrics[h]['f1'] for h in PREDICTION_HORIZONS]) if val_metrics else 0
    test_f1 = np.mean([test_metrics[h]['f1'] for h in PREDICTION_HORIZONS]) if test_metrics else 0
    
    scores_train.append({
        'f1': train_f1,
        'accuracy': np.mean([train_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]) if train_metrics else 0,
        'precision': np.mean([train_metrics[h]['precision'] for h in PREDICTION_HORIZONS]) if train_metrics else 0,
        'recall': np.mean([train_metrics[h]['recall'] for h in PREDICTION_HORIZONS]) if train_metrics else 0,
        'roc_auc': train_auc
    })
    scores_val.append({
        'f1': val_f1,
        'accuracy': np.mean([val_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]) if val_metrics else 0,
        'precision': np.mean([val_metrics[h]['precision'] for h in PREDICTION_HORIZONS]) if val_metrics else 0,
        'recall': np.mean([val_metrics[h]['recall'] for h in PREDICTION_HORIZONS]) if val_metrics else 0,
        'roc_auc': val_auc
    })
    scores_test.append({
        'f1': test_f1,
        'accuracy': np.mean([test_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]) if test_metrics else 0,
        'precision': np.mean([test_metrics[h]['precision'] for h in PREDICTION_HORIZONS]) if test_metrics else 0,
        'recall': np.mean([test_metrics[h]['recall'] for h in PREDICTION_HORIZONS]) if test_metrics else 0,
        'roc_auc': test_auc
    })
    
    # Track best model
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        best_model = copy.deepcopy(model)
    
    # Print progress every 5 epochs
    if epoch % 5 == 0:
        print(
            f"Epoch {epoch}: loss {round(train_loss, 5)}, "
            f"train {round(train_f1 * 100, 2)}%, "
            f"valid {round(val_f1 * 100, 2)}%, "
            f"test {round(test_f1 * 100, 2)}%"
        )

# Final evaluation with best model
if best_model is not None:
    model = best_model

best_train_metrics, _, _ = evaluate(model, train_loader, criterion)
best_val_metrics, _, _ = evaluate(model, val_loader, criterion)
best_test_metrics, _, _ = evaluate(model, test_loader, criterion)

best_accs = [
    {
        'f1': np.mean([best_train_metrics[h]['f1'] for h in PREDICTION_HORIZONS]),
        'accuracy': np.mean([best_train_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]),
        'precision': np.mean([best_train_metrics[h]['precision'] for h in PREDICTION_HORIZONS]),
        'recall': np.mean([best_train_metrics[h]['recall'] for h in PREDICTION_HORIZONS]),
        'true_pos': 0, 'false_pos': 0, 'true_neg': 0, 'false_neg': 0
    },
    {
        'f1': np.mean([best_val_metrics[h]['f1'] for h in PREDICTION_HORIZONS]),
        'accuracy': np.mean([best_val_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]),
        'precision': np.mean([best_val_metrics[h]['precision'] for h in PREDICTION_HORIZONS]),
        'recall': np.mean([best_val_metrics[h]['recall'] for h in PREDICTION_HORIZONS]),
        'true_pos': 0, 'false_pos': 0, 'true_neg': 0, 'false_neg': 0
    },
    {
        'f1': np.mean([best_test_metrics[h]['f1'] for h in PREDICTION_HORIZONS]),
        'accuracy': np.mean([best_test_metrics[h]['accuracy'] for h in PREDICTION_HORIZONS]),
        'precision': np.mean([best_test_metrics[h]['precision'] for h in PREDICTION_HORIZONS]),
        'recall': np.mean([best_test_metrics[h]['recall'] for h in PREDICTION_HORIZONS]),
        'true_pos': 0, 'false_pos': 0, 'true_neg': 0, 'false_neg': 0
    }
]

print(
    f"\nBest model: "
    f"train {round(best_accs[0]['f1'] * 100, 2)}%, "
    f"valid {round(best_accs[1]['f1'] * 100, 2)}%, "
    f"test {round(best_accs[2]['f1'] * 100, 2)}%"
)

## Training Graphs

In [ ]:
# Set style for good-quality plots  
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

# Create figure with multiple subplots
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.35)

epochs = range(1, len(train_loss_history) + 1)

# ============= Plot 1: Training Loss =============
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(epochs, train_loss_history, 'b-', linewidth=2, label='Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Over Epochs', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# ============= Plot 2: F1 Score Comparison =============
ax2 = fig.add_subplot(gs[1, :])
train_f1 = [s['f1'] * 100 for s in scores_train]
val_f1 = [s['f1'] * 100 for s in scores_val]
test_f1 = [s['f1'] * 100 for s in scores_test]

ax2.plot(epochs, train_f1, 'b-', linewidth=2, label='Train', marker='o', markersize=3, markevery=10)
ax2.plot(epochs, val_f1, 'g-', linewidth=2, label='Validation', marker='s', markersize=3, markevery=10)
ax2.plot(epochs, test_f1, 'r-', linewidth=2, label='Test', marker='^', markersize=3, markevery=10)

# Mark best validation epoch
best_epoch = np.argmax(val_f1)
ax2.axvline(x=best_epoch+1, color='gray', linestyle='--', alpha=0.5, label=f'Best Val (Epoch {best_epoch+1})')
ax2.scatter([best_epoch+1], [val_f1[best_epoch]], color='g', s=100, zorder=5, marker='*')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score (%)')
ax2.set_title('F1 Score Across Datasets', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='best')

# ============= Plot 3: Accuracy =============
ax3 = fig.add_subplot(gs[2, 0])
train_acc = [s['accuracy'] * 100 for s in scores_train]
val_acc = [s['accuracy'] * 100 for s in scores_val]
test_acc = [s['accuracy'] * 100 for s in scores_test]

ax3.plot(epochs, train_acc, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax3.plot(epochs, val_acc, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax3.plot(epochs, test_acc, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Accuracy (%)')
ax3.set_title('Accuracy', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best')

# ============= Plot 4: Precision =============
ax4 = fig.add_subplot(gs[2, 1])
train_prec = [s['precision'] * 100 for s in scores_train]
val_prec = [s['precision'] * 100 for s in scores_val]
test_prec = [s['precision'] * 100 for s in scores_test]

ax4.plot(epochs, train_prec, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax4.plot(epochs, val_prec, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax4.plot(epochs, test_prec, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Precision (%)')
ax4.set_title('Precision', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend(loc='best')

# ============= Plot 5: Recall =============
ax5 = fig.add_subplot(gs[2, 2])
train_rec = [s['recall'] * 100 for s in scores_train]
val_rec = [s['recall'] * 100 for s in scores_val]
test_rec = [s['recall'] * 100 for s in scores_test]

ax5.plot(epochs, train_rec, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax5.plot(epochs, val_rec, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax5.plot(epochs, test_rec, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax5.set_xlabel('Epoch')
ax5.set_ylabel('Recall (%)')
ax5.set_title('Recall', fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.legend(loc='best')

# Save the figure
plt.savefig('training_results_transformer.pdf', bbox_inches='tight', dpi=300)
plt.show()

## Best Model Evaluation Graphs

In [ ]:
# ============= Create a second figure: Final Performance Comparison =============
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot comparing final metrics
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
train_final = [best_accs[0]['accuracy']*100, best_accs[0]['precision']*100, 
               best_accs[0]['recall']*100, best_accs[0]['f1']*100]
val_final = [best_accs[1]['accuracy']*100, best_accs[1]['precision']*100, 
             best_accs[1]['recall']*100, best_accs[1]['f1']*100]
test_final = [best_accs[2]['accuracy']*100, best_accs[2]['precision']*100, 
              best_accs[2]['recall']*100, best_accs[2]['f1']*100]

x = np.arange(len(metrics_names))
width = 0.25

bars1 = ax1.bar(x - width, train_final, width, label='Train', color='#4472C4', alpha=0.8)
bars2 = ax1.bar(x, val_final, width, label='Validation', color='#70AD47', alpha=0.8)
bars3 = ax1.bar(x + width, test_final, width, label='Test', color='#ED7D31', alpha=0.8)

ax1.set_ylabel('Score (%)')
ax1.set_title('Best Model Performance Comparison', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([0, 105])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

# ROC-AUC per prediction horizon for test set
horizons = [str(h) for h in PREDICTION_HORIZONS]
test_aucs = [best_test_metrics[h]['auc'] for h in PREDICTION_HORIZONS]

bars = ax2.bar(horizons, test_aucs, color='steelblue', alpha=0.8, edgecolor='black', linewidth=1)
ax2.set_xlabel('Prediction Horizon (days)')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Test AUC-ROC by Prediction Horizon', fontweight='bold')
ax2.set_ylim([0, 1.05])
ax2.grid(True, alpha=0.3, axis='y')

for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('final_performance_comparison_transformer.pdf', bbox_inches='tight', dpi=300)
print("Saved: final_performance_comparison_transformer.pdf")
plt.show()

## ROC-AUC Graph

In [ ]:
# ============= Create a third figure: ROC AUC comparison =============
fig3, ax = plt.subplots(figsize=(8, 6))

train_auc = [s['roc_auc'] for s in scores_train]
val_auc = [s['roc_auc'] for s in scores_val]
test_auc = [s['roc_auc'] for s in scores_test]

ax.plot(epochs, train_auc, 'b-', linewidth=2, label=f"Train (Best: {max(train_auc):.4f})", 
        marker='o', markersize=3, markevery=10)
ax.plot(epochs, val_auc, 'g-', linewidth=2, label=f"Validation (Best: {max(val_auc):.4f})", 
        marker='s', markersize=3, markevery=10)
ax.plot(epochs, test_auc, 'r-', linewidth=2, label=f"Test (Best: {max(test_auc):.4f})", 
        marker='^', markersize=3, markevery=10)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('Epoch')
ax.set_ylabel('ROC AUC Score')
ax.set_title('ROC AUC Score Across Datasets', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('roc_auc_comparison_transformer.pdf', bbox_inches='tight', dpi=300)
print("Saved: roc_auc_comparison_transformer.pdf")
plt.show()

## Per-Horizon Performance

In [ ]:
# ============= Per-horizon metrics =============
fig4, axes = plt.subplots(1, len(PREDICTION_HORIZONS), figsize=(5*len(PREDICTION_HORIZONS), 5))

if len(PREDICTION_HORIZONS) == 1:
    axes = [axes]

for i, (horizon, ax) in enumerate(zip(PREDICTION_HORIZONS, axes)):
    metrics_names = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
    values = [
        best_test_metrics[horizon]['auc'],
        best_test_metrics[horizon]['accuracy'],
        best_test_metrics[horizon]['precision'],
        best_test_metrics[horizon]['recall'],
        best_test_metrics[horizon]['f1']
    ]
    
    colors = ['steelblue', 'gold', 'coral', 'lightgreen', 'mediumpurple']
    bars = ax.bar(metrics_names, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
    
    ax.set_ylabel('Score')
    ax.set_title(f'Horizon: {horizon} days', fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('per_horizon_metrics_transformer.pdf', bbox_inches='tight', dpi=300)
print("Saved: per_horizon_metrics_transformer.pdf")
plt.show()

## Save Model

In [ ]:
# Save model checkpoint
os.makedirs(CONFIG['out_dir'], exist_ok=True)

checkpoint = {
    'model_state': model.state_dict(),
    't_scaler': t_scaler,
    'config': {
        'num_nodes': num_nodes,
        'lookback_months': CONFIG['lookback_months'],
        'temporal_dim': CONFIG['temporal_dim'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'num_heads': CONFIG['num_heads'],
        'num_horizons': len(PREDICTION_HORIZONS),
        'dropout': CONFIG['dropout']
    },
    'metrics': best_test_metrics
}

save_path = os.path.join(CONFIG['out_dir'], f"transformer_unified_{file_name}.pth")
torch.save(checkpoint, save_path)
print(f"Model saved to: {save_path}")

# Print final test metrics
print("\n" + "="*60)
print("FINAL TEST METRICS")
print("="*60)
for horizon in PREDICTION_HORIZONS:
    m = best_test_metrics[horizon]
    print(f"\nHorizon {horizon} days:")
    print(f"  AUC: {m['auc']:.4f}")
    print(f"  Accuracy: {m['accuracy']:.4f}")
    print(f"  Precision: {m['precision']:.4f}")
    print(f"  Recall: {m['recall']:.4f}")
    print(f"  F1: {m['f1']:.4f}")
    print(f"  Threshold: {m['threshold']:.3f}")